In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import polars as pl

data_dir = '/content/drive/MyDrive/ml-challenge'

print("Scanning parquet files...")
q1 = pl.scan_parquet(f"{data_dir}/source1_blocked.parquet")
q2 = pl.scan_parquet(f"{data_dir}/source2_blocked.parquet")
q3 = pl.scan_parquet(f"{data_dir}/source3_blocked.parquet")

# Filter out overly dense blocking keys (> 50 occurrences) to prevent memory explosions
max_key_freq = 50
key_counts = q1.group_by("blocking_key").agg(pl.len().alias("freq"))
q1_valid_keys = q1.join(key_counts, on="blocking_key").filter(pl.col("freq") <= max_key_freq).drop("freq")

print("Streaming candidate pairs for Source 1 vs Source 2...")
query_1_2 = q1_valid_keys.join(
    q2, on="blocking_key", how="inner", suffix="_s2"
).select([
    pl.col("entity_id").alias("entity_id_1"),
    pl.col("entity_id_s2").alias("entity_id_2")
])

candidate_pairs_1_2 = query_1_2.collect(engine="streaming")
print(f"Generated {candidate_pairs_1_2.height} candidate pairs for Source 1 vs Source 2.")
candidate_pairs_1_2.write_csv(f"{data_dir}/candidate_pairs_1_2.tsv", separator="\t")

print("Streaming candidate pairs for Source 1 vs Source 3...")
query_1_3 = q1_valid_keys.join(
    q3, on="blocking_key", how="inner", suffix="_s3"
).select([
    pl.col("entity_id").alias("entity_id_1"),
    pl.col("entity_id_s3").alias("entity_id_3")
])

candidate_pairs_1_3 = query_1_3.collect(engine="streaming")
print(f"Generated {candidate_pairs_1_3.height} candidate pairs for Source 1 vs Source 3.")
candidate_pairs_1_3.write_csv(f"{data_dir}/candidate_pairs_1_3.tsv", separator="\t")

print("Success! Intermediate chunk files created safely.")

Scanning parquet files...
Streaming candidate pairs for Source 1 vs Source 2...


FileNotFoundError: No such file or directory (os error 2): /content/drive/MyDrive/ml-challenge/source1_blocked.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'join left'
	[3] 'join'
	[4] 'filter'
	[5] 'select'
	[6] 'join left'
	[7] 'join'
	[8] 'select'
	[9] 'sink'


In [ ]:
# Collect and check shape
print(q2.collect().shape)

# Fetch just the first few rows (much faster for large datasets)
print(q1.fetch(5))

(5034616, 9)
shape: (5, 9)
┌────────────┬───────────┬───────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ entity_id  ┆ business_ ┆ business_ ┆ country ┆ … ┆ address_p ┆ initials_ ┆ name_len_ ┆ blocking_ │
│ ---        ┆ name      ┆ address   ┆ ---     ┆   ┆ refix     ┆ key       ┆ bin       ┆ key       │
│ str        ┆ ---       ┆ ---       ┆ str     ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str       ┆ str       ┆         ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞════════════╪═══════════╪═══════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ S1-9257830 ┆ orelee s  ┆ 1795 west ┆ us      ┆ … ┆ 99999     ┆ bos       ┆ S         ┆ us_barb_b │
│ 39         ┆ barbersho ┆ chester   ┆         ┆   ┆           ┆           ┆           ┆ os_S_9999 │
│            ┆ p         ┆ drive,    ┆         ┆   ┆           ┆           ┆           ┆ 9         │
│            ┆           ┆ high p…   ┆         ┆   ┆           ┆

/tmp/ipykernel_4054/3347939498.py:5: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(q1.fetch(5))


In [ ]:


print("Loading master Source 1 list...")
source1_full = pl.read_csv(f"{data_dir}/train_source1.tsv", separator="\t")
all_source1_ids = source1_full.select(pl.col("entity_id").alias("source1_entity_id"))



Loading master Source 1 list...


In [ ]:
print("Processing files in  lazily formate to prevent RAM crashes... in my google colab")
# Scan intermediate files lazily instead of reading them fully into RAM
lazy_1_2 = pl.scan_csv(f"{data_dir}/candidate_pairs_1_2.tsv", separator="\t").select([
    pl.col("entity_id_1").alias("source1_entity_id"),
    pl.col("entity_id_2").alias("candidate_entity_id")
])

lazy_1_3 = pl.scan_csv(f"{data_dir}/candidate_pairs_1_3.tsv", separator="\t").select([
    pl.col("entity_id_1").alias("source1_entity_id"),
    pl.col("entity_id_3").alias("candidate_entity_id")
])

Processing files in  lazily formate to prevent RAM crashes... in my google colab


In [ ]:
# Concat, find unique pairs, and aggregate lazily
all_candidates_lazy = pl.concat([lazy_1_2, lazy_1_3]).unique()

aggregated_candidates = (
    all_candidates_lazy
    .group_by("source1_entity_id")
    .agg(pl.col("candidate_entity_id").unique().sort().str.join(","))
)

# Left join lazily with the master Source 1 list
# assign empty string corresponding to souce1 have no matching with candidates pairs
final_candidates_lazy = (
    all_source1_ids.lazy()
    .join(aggregated_candidates, on="source1_entity_id", how="left")
    .fill_null("")
    .rename({"candidate_entity_id": "candidate_entity_ids"})
)

In [ ]:
# Write out using streaming sink to avoid holding the massive combined table in RAM
print("Streaming final output directly to disk...")
output_path = f"{data_dir}/candidate_pairs.tsv"
final_candidates_lazy.sink_csv(output_path, separator="\t")

print("Successfully generated final candidate_pairs.tsv safely without crashing RAM!")

Streaming final output directly to disk...
Successfully generated final candidate_pairs.tsv safely without crashing RAM!
